# 8.3 중간고사 대비 — 실습: 유도 템플릿 3가지 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter08_3_review_diagnostic.ipynb)

책 본문: [8.3 중간고사 대비 총정리](https://smhanlab.com/book-ml/kor/ml2/chapter08/3.html)

이 노트북은 8.3절의 **유도 템플릿 3가지**를 코드로 검증하는 실습이다.
(1) 거꾸로 대입으로 푼 값이 벨만방정식의 고정점인지 잔차로 확인하고
(2) "새 MDP"에서 후보+검증 템플릿을 코드와 같은 순서로 다시 풀고
(3) 축소 사상의 기하급수 감소를 그림으로 확인한다. 시험에서
"과정을 보여주는 것"이 점수라면, 이 노트북의 각 cell은 그 과정의
참고 답안이다.


## 1. 템플릿 ①: 거꾸로 대입 — 3-state 사슬

4.3절 연습문제 2와 같은 3-state MDP(연습문제 1과 같다):
State 0 → State 1(보상 −1) → State 2(보상 −1, **터미널**), \(\gamma=0.9\).

터미널은 자기 자신으로만 전이하고 보상 0을 주는 self-loop로 표현한다 —
그래서 \(V(2)=0\)이 고정점이 된다(4.1절 노트북과 같은 설정).


In [1]:
# P[s][a] = [(prob, next_state), ...],  R[s][a] = 즉시 보상
P = [
    [[(1.0, 1)]],   # state 0 --(action 0)--> state 1
    [[(1.0, 2)]],   # state 1 --(action 0)--> state 2
    [[(1.0, 2)]],   # state 2 (터미널) --(action 0)--> state 2 (self-loop)
]
R = [
    [-1],  # state 0
    [-1],  # state 1
    [0],   # state 2 (터미널)
]
policy = [0, 0, 0]
gamma = 0.9

# 템플릿 ①: 터미널부터 거꾸로 한 칸씩 대입
V0 = 0.0                       # V(2) = 0  (터미널)
V1 = -1 + gamma * V0           # V(1) = -1
V2 = -1 + gamma * V1           # V(0) = -1.9
V_hand = [V2, V1, V0]          # [V(0), V(1), V(2)]
print("거꾸로 대입:", [round(v, 4) for v in V_hand])

거꾸로 대입: [-1.9, -1.0, 0.0]


## 2. 정확성 확인 — 벨만방정식 양변에 대입해 보기

시험 답안에 한 줄로 적을 검증: 구한 \(V=(−1.9, −1, 0)\)를
\(V(s) \stackrel{?}{=} R(s,\pi(s)) + \gamma V(s')\)의 양변에
대입하면 왼쪽과 오른쪽이 일치하는가? 상태마다 잔차를 출력한다 —
모든 잔차가 0이면 고정점이다.


In [2]:
# 벨만방정식 잔차: left - right = V(s) - [R + gamma * V(s')]
def bellman_residual(P, R, policy, gamma, V):
    res = []
    for s in range(len(P)):
        a = policy[s]
        right = R[s][a] + gamma * sum(prob * V[ns] for prob, ns in P[s][a])
        res.append(V[s] - right)
    return res

res = bellman_residual(P, R, policy, gamma, V_hand)
for s, r in enumerate(res):
    print(f"  state {s}: |V(s) - [R + gamma*V(s')]| = {abs(r):.2e}")
assert max(abs(r) for r in res) < 1e-12
print("모든 잔차 0 -> 거꾸로 대입한 V가 벨만방정식의 해(고정점)다.  [검증 통과]\n")

# 대조 실험: 대입 방향을 거꾸로 쓴 "자주 하는 실수" 버전은 고정점이 아님
#   V(1) = -1 + 0.9*V(0)  로 쓴다면? (전이 화살표 방향대로 대입한 실수)
V_wrong = [0.0] * 3
V_wrong[0] = -1 + gamma * 0      # V(0) = -1 + 0.9*V(1)... 방향이 뒤집힌 연립
V_wrong[1] = -1 + gamma * V_wrong[0]   # <- 화살표(0->1) 방향으로 대입
V_wrong[2] = 0.0
res_w = bellman_residual(P, R, policy, gamma, V_wrong)
print("방향 실수 버전의 잔차:", [f"{r:+.4f}" for r in res_w])
print("-> 잔차가 0이 아님: '대입 방향은 다음 상태가 이미 알려진 쪽'이 이유다.")

  state 0: |V(s) - [R + gamma*V(s')]| = 0.00e+00
  state 1: |V(s) - [R + gamma*V(s')]| = 0.00e+00
  state 2: |V(s) - [R + gamma*V(s')]| = 0.00e+00
모든 잔차 0 -> 거꾸로 대입한 V가 벨만방정식의 해(고정점)다.  [검증 통과]

방향 실수 버전의 잔차: ['+1.7100', '-0.9000', '+0.0000']
-> 잔차가 0이 아님: '대입 방향은 다음 상태가 이미 알려진 쪽'이 이유다.


## 3. 템플릿 ②: 후보+검증 — 강의에서 안 써본 *새* MDP

본문 §"템플릿 ②"의 예제를 그대로 코드로 옮긴다.
상태 S, G, T / \(\gamma=0.9\):

- **S**: run(0.7 확률 G, 0.3 확률 S, 보상 0) 또는 rest(S에 남음, 보상 −1)
- **G**: collect(T로, 보상 +3)
- **T**: 터미널

순서: (1) 후보 추측 → (2) 후보를 **정답인 것처럼 가정하고** 방정식
해소 → (3) 나머지 행동을 실제로 계산해서 후보의 행동을 **검증**한다.


In [3]:
gamma = 0.9

# --- Step 1 (후보 추측) ---------------------------------------------
# +3을 받는 유일한 길은 G에서 collect:  Q*(G, collect) = 3 + 0.9*0 = 3
Q_G_collect = 3 + gamma * 0.0
# 후보: run@S, collect@G
print(f"Step 1  후보: run@S, collect@G  (Q(G,collect) = {Q_G_collect})")

# --- Step 2 (해 구하기: run@S가 최선인 것을 가정) --------------------
# V*(S) = 0.9 * (0.7*V*(G) + 0.3*V*(S)),  V*(G) = Q(G,collect) = 3
# 자기 자신으로의 확률 0.9*0.3 = 0.27 때문에 V*(S)가 자기 자신에 대해 나옴:
#   V_S = 0.9*(0.7*V_G + 0.3*V_S)  ->  (1 - 0.9*0.3)*V_S = 0.9*0.7*V_G
V_G = Q_G_collect
V_S = gamma * (0.7 * V_G) / (1 - gamma * 0.3)
print(f"Step 2  V*(S) = 0.9*0.7*3 / (1 - 0.9*0.3) = {189/73:.4f}  (정확한 값 189/73 = {189/73:.4f})")

# --- Step 3 (검증: run이 정말 rest보다 나은가) ------------------------
Q_S_run  = V_S                                  # run의 Q값 = 가정 하에 푼 V(S)
Q_S_rest = -1 + gamma * V_S                     # rest의 Q값을 실제로 계산
print(f"Step 3  Q(S,run)  = {Q_S_run:.3f}")
print(f"        Q(S,rest) = -1 + 0.9*V*(S) = {Q_S_rest:.3f}")
assert Q_S_run > Q_S_rest
print("Q(run) > Q(rest) -> 후보가 성립.  답: V*(S) = 2.589, V*(G) = 3  [검증 통과]")

Step 1  후보: run@S, collect@G  (Q(G,collect) = 3.0)
Step 2  V*(S) = 0.9*0.7*3 / (1 - 0.9*0.3) = 2.5890  (정확한 값 189/73 = 2.5890)
Step 3  Q(S,run)  = 2.589
        Q(S,rest) = -1 + 0.9*V*(S) = 1.330
Q(run) > Q(rest) -> 후보가 성립.  답: V*(S) = 2.589, V*(G) = 3  [검증 통과]


### (부록) 검증이 형식이 아니라 안전장치인 이유 — Step 1을 일부러 틀려 보기

Step 1의 추측이 틀리면 Step 2의 값은 "틀린 정책을 최적인 것처럼
푼" 값이고, Step 3의 부등호가 뒤집힌다. 후보를 **rest@S**로 바꿔서
동일한 순서(가정 → 해소 → 검증)를 돌리면, rest@S가 실제로 최적이기
때문에 검증에서 run 쪽이 더 좋아 나와야 한다 — "후보가 틀렸으니 다시
추측하라"는 신호다.


In [4]:
# (부록) 일부러 틀린 후보 rest@S로 Step 2를 돌린다:
# rest@S가 최선인 것처럼 가정하면  V(S) = -1 + 0.9*V(S)
V_S_rest = -1 / (1 - gamma)
# 이제 S에서 run의 Q를 *이* V 기준으로 실제로 계산:
#   Q(S,run) = 0.9*(0.7*V(G) + 0.3*V(S)),  V(G) = 3
Q_S_run_vs = gamma * (0.7 * 3.0 + 0.3 * V_S_rest)
print(f"후보 rest@S 가정 하에:  V(S) = {V_S_rest:.1f}")
print(f"  Q(S,rest) = {V_S_rest:.1f}")
print(f"  Q(S,run)  = 0.9*(0.7*3 + 0.3*(-10)) = {Q_S_run_vs:.3f}")
print("-> Q(S,run) > Q(S,rest): 검증이 후보를 기각. Step 1로 돌아가야 한다는 신호.\n")

# 4.2절 정책반복과의 연결: pi_0 = rest@S에서 시작하면
print("정책반복 연결: pi_0 = rest@S")
print(f"  평가:  V^pi(S) = -1/(1-0.9) = {V_S_rest:.0f}")
print(f"  개선:  Q^pi(S,run) = {Q_S_run_vs:.2f} > {V_S_rest:.0f}  -> run으로 교체")
print(f"  재평가: run@S의 V(S) = {V_S:.4f}  (Step 2의 결과와 동일, 1회 반복)")

후보 rest@S 가정 하에:  V(S) = -10.0
  Q(S,rest) = -10.0
  Q(S,run)  = 0.9*(0.7*3 + 0.3*(-10)) = -0.810
-> Q(S,run) > Q(S,rest): 검증이 후보를 기각. Step 1로 돌아가야 한다는 신호.

정책반복 연결: pi_0 = rest@S
  평가:  V^pi(S) = -1/(1-0.9) = -10
  개선:  Q^pi(S,run) = -0.81 > -10  -> run으로 교체
  재평가: run@S의 V(S) = 2.5890  (Step 2의 결과와 동일, 1회 반복)


## 4. 템플릿 ③: 축소 사상 — 기하급수 감소

"오차가 1% 이하가 되기까지 몇 번의 반복이 필요한가?" —
\(\gamma^n < 0.01\)을 \(n\)에 대해 풀면
\(n > \ln 0.01/\ln 0.9 \approx 44\).
본문 표의 수치(\(\gamma^n\)의 n=5,10,20,50)를 검증하고,
감소 곡선을 로그 스케일로 그린다.


In [5]:
import math
gamma = 0.9

# 본문 표 검증
for n in (5, 10, 20, 50):
    print(f"  n={n:>2}:  gamma^n = {gamma**n:.4f}")

# 1% 기준 반복 수
n_1pct = math.ceil(math.log(0.01) / math.log(gamma))
print(f"\ngamma^n < 0.01 인 가장 작은 n = {n_1pct}  (본문: 약 44)")

import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

ns = range(1, 61)
decs = [gamma**n for n in ns]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(ns, decs, color="#4a90d9", lw=2, label=r"$\|V_n - V^*\| \leq \gamma^n \|V_0 - V^*\|$")
ax.axhline(0.01, color="gray", ls="--", lw=1)
ax.axvline(n_1pct, color="gray", ls=":", lw=1)
ax.text(n_1pct + 0.5, 0.0105, f"n = {n_1pct} (1% threshold)", fontsize=9, color="gray")
for n in (5, 10, 20, 50):
    ax.plot([n], [gamma**n], "o", color="#d9534f", ms=6)
ax.set_xlabel("Number of iterations n")
ax.set_ylabel(r"$\gamma^n$ (log scale)")
ax.set_title(r"Contraction mapping: initial error decays geometrically by a factor of $\gamma^n$ (a straight line on the log axis = geometric decay)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig("ch08_3_contraction_decay_local.svg", bbox_inches="tight")
plt.show()
print("\n-> 로그 축에서 직선: '기하급수'의 시각적 의미. gamma=0.99라면 1%까지 n=458.")

  n= 5:  gamma^n = 0.5905
  n=10:  gamma^n = 0.3487
  n=20:  gamma^n = 0.1216
  n=50:  gamma^n = 0.0052

gamma^n < 0.01 인 가장 작은 n = 44  (본문: 약 44)



-> 로그 축에서 직선: '기하급수'의 시각적 의미. gamma=0.99라면 1%까지 n=458.


## 정리 — 시험 답안의 3줄

1. **거꾸로 대입**(§1~2): 전이가 한 방향으로만 흐를 때, 터미널부터 대입하면
   연립방정식을 풀지 않고 풀린다. 답안 마지막 줄에 "대입 방향은 전이
   화살표의 반대(다음 상태가 이미 알려진 쪽)"와 "벨만방정식 양변에 대입
   해 잔차가 0" 한 줄을 꼭 쓴다.
2. **후보+검증**(§3): \(\max\)을 알지 못하면 펼칠 수 없으니,
   (1) 후보 추측 → (2) 가정 하에 해소 → (3) 검증 순서가 **의무**다.
   검증의 부등호가 뒤집히면 "후보가 틀렸다"는 신호.
3. **축소 사상 논증**(§4): 고정점 전제를 명시하고 4줄로
   \(\|V_n - V^*\| \le \gamma^n \|V_0 - V^*\| \to 0\).
   "1%까지 몇 번인가" = \(n > \ln 0.01/\ln \gamma\).
